In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# 定义CNN
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Block1  
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2)  # 28x28 -> 14x14

        # Block2
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2)  # 14x14 -> 7x7

        # 全连接
        self.fc1 = nn.Linear(16 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

        # 展平
        self.flatten = nn.Flatten()

    def forward(self, x):
        # Block1
        x = F.relu(self.conv1(x))   
        x = self.pool1(x)

        # Block2
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
    
        # Flatten + FC
        x = self.flatten(x)  # (16 * 7 * 7)
        x = F.relu(self.fc1(x))  # 128
        x = self.fc2(x)   # 10
        return x

- 当需要参数时，需要在init中声明，只能用nn.Module  
但relu和max_pool不需要参数学习，可以直接用F.relu F.max_pool2d来更简洁
- 几乎每层卷积/全连接后面都要跟一个F.relu
- 特征图尺寸计算：(H + 2P - K) / S + 1

In [ ]:
# 定义训练循环
def evaluate(model, loader):
    model.eval()  # 将模型切换到评估模式
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()  # 加上item()是为了将Tensor转为Python的int
            total += y.size(0)  # 每一个batch的数量
    return correct / total

def train(model, train_loader, test_loader, epochs=5, lr=0.01):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = {'loss': [], 'acc': []}

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            # 核心5步
            optimizer.zero_grad()  # 先清原有梯度
            preds = model(x)  # 得到模型预测结果
            loss = criterion(preds, y)  # 求损失
            loss.backward()  # 反向传播算出参数梯度
            optimizer.step()  # 参数更新

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        acc = evaluate(model, test_loader)
        history['loss'].append(avg_loss)
        history['acc'].append(acc)

        print(f'Epoch:{epoch + 1} / {epochs} | Loss:{avg_loss:.4f} | Acc:{acc:.4f}')

    return history

model = CNN().to(device)
history = train(model, train_loader, test_loader, epochs=5, lr=0.01)